# 02 · Training — HPO fan-out and a model Artifact

Chapter 1 produced the `review-dataset`. Now find the best hyperparameters for the review
classifier by evaluating many configurations **in parallel**, then train the winner and
publish it as the **`review-classifier`** model Artifact that Chapters 3 and 4 consume.

The model is a TF-IDF + LogisticRegression classifier (CPU, trains in milliseconds) so the
whole sweep fits in a workshop slot. The scale-up note at the end shows how the same shape
becomes a GPU / distributed training job unchanged.

> **Note on types:** notebook task I/O uses built-in types (here `list[float]`); the matching
> script `workflows/02_hpo.py` uses a `@dataclass TrialResult` for readability.

In [ ]:
from pathlib import Path
import asyncio
import pickle

import flyte
from flyte.io import File

# connect to the cluster
flyte.init_from_config(Path('.flyte') / 'config.yaml')

image = flyte.Image.from_debian_base().with_pip_packages('scikit-learn', 'unionai-reuse>=0.1.9')

## Step 1 · A pool of HPO trial workers

In [ ]:
trial_env = flyte.TaskEnvironment(
    name='hpo_trial',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
    # warm pool that the trial workers share
    reusable=flyte.ReusePolicy(replicas=(1, 3), concurrency=4, idle_ttl=90),
)

driver_env = flyte.TaskEnvironment(
    name='hpo_driver',
    image=image,
    resources=flyte.Resources(cpu=1, memory='512Mi'),
    # the driver orchestrates the trials
    depends_on=[trial_env],
)

## Step 2 · One trial = one action, fanned out

`optimize` launches the whole grid at once with `asyncio.gather` inside `flyte.group`, then
returns the best config as `[C, ngram, score]` by 3-fold cross-validated accuracy.

In [ ]:
_POS = ['love it', 'works great', 'excellent quality', 'highly recommend',
        'fast and reliable', 'best purchase ever', 'so happy with this']
_NEG = ['broke immediately', 'terrible support', 'waste of money', 'very disappointed',
        'stopped working', 'would not buy again', 'cheap and flimsy']

def _synth(n: int) -> tuple[list[str], list[str]]:
    texts, labels = [], []
    for i in range(n):
        if i % 2 == 0:
            texts.append(_POS[i % len(_POS)] + (' ' + _NEG[i % len(_NEG)] if i % 7 == 0 else ''))
            labels.append('positive')
        else:
            texts.append(_NEG[i % len(_NEG)] + (' ' + _POS[i % len(_POS)] if i % 7 == 0 else ''))
            labels.append('negative')
    return texts, labels

In [ ]:
# one hyperparameter combo = one action, scored by 3-fold cross-validation
@trial_env.task
async def evaluate_config(c: float, ngram: int, n_reviews: int) -> float:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import Pipeline

    texts, labels = _synth(n_reviews)
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, ngram))),
        ('clf', LogisticRegression(C=c, max_iter=1000)),
    ])
    score = float(cross_val_score(pipe, texts, labels, cv=3).mean())
    print(f'trial C={c} ngram={ngram} -> cv_accuracy={score:.4f}')
    return score

@driver_env.task
async def optimize(n_reviews: int = 300) -> list[float]:
    grid = [(c, ng) for c in (0.1, 1.0, 10.0) for ng in (1, 2)]
    # launch the whole grid of trials at once
    with flyte.group('hpo-sweep'):
        scores = await asyncio.gather(*(evaluate_config(c, ng, n_reviews) for c, ng in grid))
    
    # pick the winner by cross-val accuracy
    best_i = max(range(len(scores)), key=lambda i: scores[i])
    best_c, best_ng = grid[best_i]
    
    print(f'WINNER: C={best_c} ngram={best_ng} cv_accuracy={scores[best_i]:.4f}')
    return [best_c, float(best_ng), scores[best_i]]

## Step 3 · Run the sweep

In [ ]:
# fan out the sweep on the cluster
hpo_run = flyte.run(optimize)
print('Run URL:', hpo_run.url)
hpo_run.wait()
best_c, best_ngram, best_score = hpo_run.outputs().o0
best_ngram = int(best_ngram)
print(f'best: C={best_c} ngram={best_ngram} cv_accuracy={best_score:.4f}')

## Step 4 · Train the winner and publish the model Artifact

`train` fits the classifier on the full dataset and returns it via `flyte.artifacts.new`
under the name **`review-classifier`** — giving you dataset -> model -> serving lineage in
the UI. (Run-input names must be lowercase, so the regularization strength is `c`.)

In [ ]:
train_env = flyte.TaskEnvironment(
    name='train_model',
    image=flyte.Image.from_debian_base().with_pip_packages('scikit-learn'),
    resources=flyte.Resources(cpu=2, memory='1Gi'),
)

# fit the winner and publish it as the review-classifier Artifact
@train_env.task(produces_artifacts=True)
async def train(c: float = 1.0, ngram: int = 2, n_reviews: int = 300) -> File:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    texts, labels = _synth(n_reviews)
    model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, ngram))),
        ('clf', LogisticRegression(C=c, max_iter=1000)),
    ])
    model.fit(texts, labels)
    acc = float(model.score(texts, labels))
    print(f'trained review-classifier: C={c} ngram={ngram} train_accuracy={acc:.4f}')
    with open('/tmp/review_classifier.pkl', 'wb') as f:
        pickle.dump(model, f)
    model_file = await File.from_local('/tmp/review_classifier.pkl')
    # named, versioned model Artifact (Chapters 3 and 4 consume it)
    return flyte.artifacts.new(
        model_file,
        flyte.artifacts.Metadata(
            name='review-classifier',
            description='TF-IDF + LogisticRegression review sentiment classifier',
            attrs={'C': str(c), 'ngram': str(ngram), 'train_accuracy': f'{acc:.4f}'},
        ),
    )

In [ ]:
# train with the winning hyperparameters
train_run = flyte.run(train, c=best_c, ngram=best_ngram)
print('Run URL:', train_run.url)
train_run.wait()
print('phase:', train_run.phase)
model_file = train_run.outputs().o0
print('model Artifact ->', model_file.path)

## Scaling up (reference)

The task *interface* — inputs, the model Artifact output — stays identical as you scale the
training itself:

- **GPU fine-tune:** give `train_env` `flyte.Resources(gpu='A100:1', ...)` and swap the
  sklearn fit for a transformer fine-tune.
- **Multi-node distributed:** attach the `Elastic` plugin so Flyte launches `torchrun` across
  nodes (FSDP, streaming datasets) — see the distributed-pretraining tutorial.

Because the winner and the model are just task I/O, none of the orchestration around them
changes.

Next: **`03_batch_inference.ipynb`** — score a large corpus with this model.